In [1]:
library(scMINER)
library(dplyr)
library(ggplot2)
library(anndata)
library(hdf5r)

Loading required package: Biobase

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Loading required package: Matrix

Loading required package: ggplot2


Attaching package: ‘dplyr’


The following object is masked from ‘package:B

In [2]:
scminer_dir <- createProjectSpace(project_dir = "scminer_project_files", project_name = "Human_Kidney_Cancer_Interface", do.unlink = TRUE)

A folder of the specified project name was found and has been cleaned up in the specified project directory
The project space has been successfully created: scminer_project_files/Human_Kidney_Cancer_Interface !


In [3]:
adata_decomp <- read_h5ad("spotiphy_project_files/Human Kidney Cancer Interface/ST_decomposition.h5ad")                      

In [4]:
class(adata_decomp)

[1] "AnnDataR6" "R6"

In [5]:
# Convert to dense numpy array then to R
expr_matrix <- reticulate::py_to_r(
  adata_decomp$.__enclos_env__$private$.anndata$X
)

# Then make it sparse in R
expr_matrix <- Matrix::Matrix(expr_matrix, sparse = TRUE)

# Get the transpose
expr_matrix <- Matrix::t(expr_matrix)

expr_matrix

10827 x 22430 sparse Matrix of class "dgCMatrix"
                                                                                     
   [1,]     .        .            .            .                .        .           
   [2,]     .        .            3.615161e+01 .                .        .           
   [3,]     .        .            .            .                .        .           
   [4,]   492.106724 .            1.330104e+02 .              775.596411 .           
   [5,]     .        .            1.565807e+00 .                .        .           
   [6,]   298.261141 6.728170e+02 .            .              219.363547 .           
   [7,]     .        .            .            .                .        .           
   [8,]     .        .            .            .                .        .           
   [9,]     .        .            .            .                .        .           
  [10,]   298.304678 3.626609e+02 1.709583e+02 .                .        .           
  [11

In [6]:
rownames(expr_matrix) <- make.names(unlist(adata_decomp$var_names), unique = TRUE)
colnames(expr_matrix) <- unlist(adata_decomp$obs_names)

In [7]:
raw.eset <- createSparseEset(
  input_matrix = expr_matrix,
  cellData = adata_decomp$obs,
  featureData = adata_decomp$var,
  projectID    = "Human Kidney Cancer Interface",
  addMetaData  = TRUE
)

raw.eset
# access metadata using pData(raw.eset) and fData(raw.eset)

Creating sparse eset from the input_matrix ...
	Adding meta data based on input_matrix ...


Warning message in createSparseEset(input_matrix = expr_matrix, cellData = adata_decomp$obs, :
“The input_matrix might not be a raw count matrix, since non-integer values were found. The 'nUMI' column is only valid when the values in input_matrix are raw UMI counts.”


Done! The sparse eset has been generated: 10827 genes, 22430 cells.


SparseExpressionSet (storageMode: environment)
assayData: 10827 features, 22430 samples 
  element names: exprs 
protocolData: none
phenoData
  sampleNames: 0 1 ... 22429 (22430 total)
  varLabels: cell_type spot_name ... CellID (11 total)
  varMetadata: labelDescription
featureData
  featureNames: ENSG00000018869 ENSG00000130775 ... ENSG00000211445
    (10827 total)
  fvarLabels: SYMBOL nCell
  fvarMetadata: labelDescription
experimentData: use 'experimentData(object)'
Annotation:  

In [15]:
drawSparseEsetQC(
  input_eset       = raw.eset,
  output_html_file = file.path(scminer_dir, "PLOT/qc_report.html"),
  overwrite        = TRUE
)

Checkinig the output html file ...
Checking the availability of the 5 metrics ('nCell', 'nUMI', 'nFeature', 'pctMito', 'pctSpikeIn') used for quanlity control ...
	Checking passed! All 5 metrics are available.




processing file: qc_report.Rmd



1/35                          
2/35 [setup]                  


ERROR: [1m[33mError[39m in `library()`:[22m
[33m![39m there is no package called 'plotly'



Quitting from lines 9-26 [setup] (qc_report.Rmd)



In [8]:
filtered.eset <- filterSparseEset(
    raw.eset,
    filter_mode = "auto",
    filter_type = "both"
)

Checking the availability of the 5 metrics ('nCell', 'nUMI', 'nFeature', 'pctMito', 'pctSpikeIn') used for filtration ...
Checking passed! All 5 metrics are available.
Filtration is done!


Filtration Summary:

	10680/10827 genes passed!

	15277/22430 cells passed!


For more details:
	Gene filtration statistics:
		Metrics		nCell
		Cutoff_Low	112
		Cutoff_High	Inf
		Gene_total	10827
		Gene_passed	10680(98.64%)
		Gene_failed	147(1.36%)

	Cell filtration statistics:
		Metrics		nUMI		nFeature	pctMito		pctSpikeIn	Combined
		Cutoff_Low	999999		287		0		0		NA
		Cutoff_High	1000000		Inf		0.0000		0.0000		NA
		Cell_total	22430		22430		22430		22430		22430
		Cell_passed	15291(68.17%)	22416(99.94%)	22430(100.00%)	22430(100.00%)	15277(68.11%)
		Cell_failed	7139(31.83%)	14(0.06%)	0(0.00%)	0(0.00%)	7153(31.89%)




In [9]:
filtered.eset

SparseExpressionSet (storageMode: environment)
assayData: 10680 features, 15277 samples 
  element names: exprs 
protocolData: none
phenoData
  sampleNames: 0 1 ... 22429 (15277 total)
  varLabels: cell_type spot_name ... CellID (11 total)
  varMetadata: labelDescription
featureData
  featureNames: ENSG00000018869 ENSG00000130775 ... ENSG00000211445
    (10680 total)
  fvarLabels: SYMBOL nCell
  fvarMetadata: labelDescription
experimentData: use 'experimentData(object)'
Annotation:  

In [10]:
log2cpm.eset <- normalizeSparseEset(
    filtered.eset,
    log_base        = 2,
    log_pseudoCount = 1
)

Done! The data matrix of eset has been normalized and log-transformed!
The returned eset contains: 10680 genes, 15277 cells.


In [76]:
ncol(log2cpm.eset)

Samples 
  15277

In [11]:
saveRDS(log2cpm.eset, file.path(scminer_dir, "DATA/log2cpm.rds"))

In [79]:
# ── 7. Generate MICA input ───────────────────────────────────────────────────
mica_input_file <- file.path(scminer_dir, "MICA/micaInput.h5ad")
generateMICAinput(
  input_eset  = log2cpm.eset,
  output_file = mica_input_file,
  overwrite   = TRUE
)


Writing MICA input to: scminer_project_files/Human_Kidney_Cancer_Interface/MICA/micaInput.h5ad 

For dataset with more than 5k cells, MICA GE mode is recommended.
Suggested command line is:

 With the default resolusion:
	mica ge -i scminer_project_files/Human_Kidney_Cancer_Interface/MICA/micaInput.h5ad -o project_space/MICA/mica_output_dir -res 1.822 -nw 8

 Or with a range of resolutions:
	mica ge -i scminer_project_files/Human_Kidney_Cancer_Interface/MICA/micaInput.h5ad -o project_space/MICA/mica_output_dir -minr 1.0 -maxr 2.0 -ss 0.1 -nw 8 

    Where options represent:
        -res:   the number of communities (default: 1.822)
        -minr:  the minimum number of communities to sweep (default: 1.822)
        -maxr:  the maximum number of communities to sweep (default: 1.822)
        -ss:    the step size to sweep number of communities from the mimimum to the maximum (default: 1)
        -nw:    the number of workers to run in parallel (default: 1, suggested: 25)
        -annef: t

In [43]:
install.packages("rjson")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [46]:
mica_input <- read.table(mica_input_file, header = TRUE, sep = "\t")

In [47]:
mica_input

X,X0,X1,X2,X3,X4,X5,X6,X7,X10,⋯,X22420,X22421,X22422,X22423,X22424,X22425,X22426,X22427,X22428,X22429
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ENSG00000018869,0.000000,0.000000,0.0000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,⋯,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000130775,0.000000,0.000000,5.2153528,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,⋯,0.000000,0.000000,10.56150,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000134759,0.000000,0.000000,0.0000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.303816,⋯,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000090487,8.945756,0.000000,7.0662015,0.000000,9.601021,0.000000,0.000000,8.922452,0.000000,⋯,0.000000,0.000000,0.00000,9.010708,0.000000,0.000000,0.000000,0.000000,7.845454,0.000000
ENSG00000171234,0.000000,0.000000,1.3594127,0.000000,0.000000,0.000000,0.000000,3.038252,3.738533,⋯,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000074054,8.225261,9.396213,0.0000000,0.000000,7.783742,0.000000,0.000000,0.000000,0.000000,⋯,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000049249,0.000000,0.000000,0.0000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,⋯,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000117399,0.000000,0.000000,0.0000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,⋯,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000159692,8.225471,8.506450,7.4259152,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,⋯,0.000000,9.153635,0.00000,0.000000,9.572001,0.000000,0.000000,0.000000,0.000000,8.827217


In [41]:
mica_input_file

[1] "scminer_project_files/Human_Kidney_Cancer_Interface/MICA/micaInput.txt"

In [13]:
mica_out_dir <- file.path(scminer_dir, "MICA/GE")
n_cells <- ncol(log2cpm.eset)

if (n_cells < 5000) {
  # MDS mode (K-Means) — better for small datasets
  output <- system(paste(
    "mica mds",
    "-i", mica_input_file,
    "-o", file.path(scminer_dir, "MICA/MDS"),
    "-nck 5 6 7 8 9 10"
  ), intern= TRUE)
} else {
  # GE mode (Louvain) — better for large datasets
  output <- system(paste(
    "mica ge",
    "-i", mica_input_file,
    "-o", mica_out_dir,
    "-minr 0.1 -maxr 9.0 -ss 0.05 -nw 40"
  ), intern= TRUE)
}
cat(output, sep="\n")

In [49]:
mica_output_file <- file.path(mica_out_dir, "clustering_UMAP_euclidean_20_2.15.txt")

In [50]:
micaResult <- read.table(mica_output_file, header = TRUE, sep = "\t", quote = "", stringsAsFactors = FALSE, row.names = 1)

In [51]:
rownames(micaResult)

[1] "0"    "1"    "2"    "3"    "4"    "5"    "6"    "7"    "8"    "9"   
 [11] "10"   "11"   "12"   "13"   "14"   "15"   "16"   "17"   "18"   "19"  
 [21] "20"   "21"   "22"   "23"   "24"   "25"   "26"   "27"   "28"   "29"  
 [31] "30"   "31"   "32"   "33"   "34"   "35"   "36"   "37"   "38"   "39"  
 [41] "40"   "41"   "42"   "43"   "44"   "45"   "46"   "47"   "48"   "49"  
 [51] "50"   "51"   "52"   "53"   "54"   "55"   "56"   "57"   "58"   "59"  
 [61] "60"   "61"   "62"   "63"   "64"   "65"   "66"   "67"   "68"   "69"  
 [71] "70"   "71"   "72"   "73"   "74"   "75"   "76"   "77"   "78"   "79"  
 [81] "80"   "81"   "82"   "83"   "84"   "85"   "86"   "87"   "88"   "89"  
 [91] "90"   "91"   "92"   "93"   "94"   "95"   "96"   "97"   "98"   "99"  
[101] "100"  "101"  "102"  "103"  "104"  "105"  "106"  "107"  "108"  "109" 
[111] "110"  "111"  "112"  "113"  "114"  "115"  "116"  "117"  "118"  "119" 
[121] "120"  "121"  "122"  "123"  "124"  "125"  "126"  "127"  "128"  "129" 
[131] "130"  "131"  "132"  "133"  "134"  "135"  "136"  "137"  "138"  "139" 
[141] "140"  "141"  "142"  "143"  "144"  "145"  "146"  "147"  "148"  "149" 
[151] "150"  "151"  "152"  "153"  "154"  "155"  "156"  "157"  "158"  "159" 
[161] "160"  "161"  "162"  "163"  "164"  "165"  "166"  "167"  "168"  "169" 
[171] "170"  "171"  "172"  "173"  "174"  "175"  "176"  "177"  "178"  "179" 
[181] "180"  "181"  "182"  "183"  "184"  "185"  "186"  "187"  "188"  "189" 
[191] "190"  "191"  "192"  "193"  "194"  "195"  "196"  "197"  "198"  "199" 
[201] "200"  "201"  "202"  "203"  "204"  "205"  "206"  "207"  "208"  "209" 
[211] "210"  "211"  "212"  "213"  "214"  "215"  "216"  "217"  "218"  "219" 
[221] "220"  "221"  "222"  "223"  "224"  "225"  "226"  "227"  "228"  "229" 
[231] "230"  "231"  "232"  "233"  "234"  "235"  "236"  "237"  "238"  "239" 
[241] "240"  "241"  "242"  "243"  "244"  "246"  "247"  "248"  "249"  "250" 
[251] "251"  "256"  "257"  "258"  "263"  "264"  "265"  "4955" "4956" "4957"
[261] "4964" "4969" "4973" "4975" "9487" "9488" "9489" "9490" "9491" "9492"
[271] "9494" "9495" "9496" "9497" "9500" "9501" "9503" "9504" "9505" "9506"
[281] "9507" "9508" "9509" "9511" "9513" "9514" "9516" "9517" "9520" "9525"
[291] "9526" "9527" "9528" "9529" "9530" "9534" "9537" "9538" "9539" "9540"
[301] "9542" "9543" "9544" "9545" "9546" "9548" "9550" "9551" "9552" "9553"
[311] "9555" "9556" "9557" "9559" "9560" "9561" "9562" "9563" "9566" "9567"
[321] "9569" "9580" "9581" "9582" "9583" "9584" "9585" "9586" "9587" "9590"
[331] "9591" "9592" "9593" "9594" "9595" "9596" "9597"

In [38]:
colnames(log2cpm.eset)

[1] "0"     "1"     "2"     "3"     "4"     "5"     "6"     "7"     "10"   
   [10] "11"    "12"    "13"    "14"    "15"    "16"    "19"    "20"    "21"   
   [19] "22"    "23"    "24"    "25"    "26"    "27"    "28"    "29"    "30"   
   [28] "31"    "32"    "35"    "36"    "37"    "38"    "39"    "41"    "42"   
   [37] "43"    "44"    "45"    "46"    "47"    "48"    "49"    "50"    "51"   
   [46] "52"    "53"    "54"    "55"    "56"    "57"    "58"    "59"    "60"   
   [55] "61"    "62"    "65"    "66"    "67"    "68"    "69"    "70"    "71"   
   [64] "72"    "73"    "74"    "75"    "76"    "77"    "78"    "79"    "80"   
   [73] "81"    "82"    "83"    "84"    "85"    "86"    "87"    "88"    "91"   
   [82] "92"    "93"    "96"    "97"    "98"    "99"    "100"   "102"   "103"  
   [91] "104"   "105"   "106"   "107"   "108"   "109"   "110"   "111"   "112"  
  [100] "113"   "114"   "115"   "116"   "117"   "118"   "119"   "122"   "123"  
  [109] "124"   "125"   "126"   "127"   "128"   "129"   "130"   "131"   "132"  
  [118] "133"   "134"   "135"   "136"   "137"   "138"   "139"   "140"   "141"  
  [127] "142"   "143"   "144"   "146"   "147"   "148"   "151"   "153"   "154"  
  [136] "155"   "156"   "157"   "158"   "159"   "160"   "161"   "162"   "163"  
  [145] "164"   "165"   "166"   "167"   "168"   "169"   "172"   "173"   "174"  
  [154] "175"   "176"   "177"   "182"   "183"   "186"   "187"   "188"   "189"  
  [163] "190"   "191"   "192"   "193"   "194"   "195"   "197"   "199"   "200"  
  [172] "201"   "202"   "203"   "204"   "205"   "206"   "207"   "208"   "211"  
  [181] "213"   "214"   "215"   "216"   "217"   "218"   "219"   "220"   "221"  
  [190] "222"   "223"   "224"   "225"   "226"   "227"   "228"   "230"   "232"  
  [199] "233"   "234"   "238"   "239"   "240"   "241"   "242"   "243"   "244"  
  [208] "245"   "246"   "247"   "248"   "249"   "251"   "252"   "253"   "254"  
  [217] "255"   "256"   "257"   "258"   "259"   "260"   "261"   "262"   "263"  
  [226] "264"   "265"   "268"   "269"   "270"   "271"   "272"   "273"   "274"  
  [235] "275"   "276"   "277"   "278"   "279"   "280"   "281"   "284"   "285"  
  [244] "286"   "287"   "288"   "290"   "291"   "292"   "293"   "294"   "295"  
  [253] "296"   "297"   "298"   "299"   "300"   "301"   "302"   "303"   "304"  
  [262] "305"   "306"   "310"   "311"   "312"   "313"   "318"   "319"   "320"  
  [271] "321"   "322"   "323"   "324"   "325"   "326"   "327"   "328"   "329"  
  [280] "330"   "331"   "334"   "335"   "336"   "337"   "338"   "339"   "340"  
  [289] "341"   "342"   "343"   "344"   "345"   "348"   "354"   "355"   "356"  
  [298] "363"   "364"   "365"   "366"   "367"   "368"   "369"   "370"   "371"  
  [307] "388"   "389"   "390"   "391"   "392"   "393"   "394"   "395"   "396"  
  [316] "397"   "398"   "399"   "400"   "401"   "402"   "403"   "404"   "405"  
  [325] "406"   "407"   "408"   "413"   "414"   "415"   "417"   "418"   "419"  
  [334] "420"   "421"   "422"   "427"   "428"   "429"   "430"   "431"   "432"  
  [343] "433"   "434"   "435"   "436"   "437"   "438"   "440"   "449"   "450"  
  [352] "451"   "452"   "453"   "454"   "455"   "456"   "457"   "467"   "468"  
  [361] "469"   "470"   "471"   "472"   "478"   "479"   "480"   "481"   "482"  
  [370] "483"   "484"   "485"   "486"   "487"   "488"   "489"   "490"   "491"  
  [379] "492"   "493"   "494"   "495"   "496"   "500"   "501"   "502"   "503"  
  [388] "505"   "506"   "507"   "508"   "509"   "510"   "514"   "515"   "516"  
  [397] "517"   "518"   "519"   "520"   "521"   "522"   "523"   "524"   "525"  
  [406] "526"   "527"   "528"   "533"   "534"   "535"   "536"   "537"   "538"  
  [415] "539"   "544"   "545"   "553"   "566"   "567"   "568"   "569"   "570"  
  [424] "571"   "572"   "575"   "576"   "579"   "580"   "581"   "582"   "583"  
  [433] "584"   "589"   "590"   "591"   "592"   "593"   "594"   "595"   "599"  
  [442] "600"   "601"   "602"   "603"   "604"   "620"   "621"   "624"   "625"  
  [4

In [55]:
summary(present_nUMI)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
                                                

In [74]:
# Find missing cells
all_cells <- colnames(log2cpm.eset)
clustered_cells <- rownames(micaResult)
missing_cells <- setdiff(all_cells, clustered_cells)

length(all_cells)
length(clustered_cells)
length(missing_cells)  # ~13430 missing

[1] 15277

[1] 337

[1] 14997

In [66]:
missing_nUMI  <- pData(log2cpm.eset)[missing_cells, "cell_type"]
present_nUMI  <- pData(log2cpm.eset)[clustered_cells, "cell_type"]

summary(missing_nUMI)
summary(present_nUMI)

B-cell         EC     Epi_PT Epi_non-PT      Fibro       Mast    Myeloid 
        17       3141        152       2577       1441        182       2286 
        NK     Plasma        RCC     T-cell        pDC 
        91         19       4490        486        115

B-cell         EC     Epi_PT Epi_non-PT      Fibro       Mast    Myeloid 
       218          1          6          0         56          0          0 
        NK     Plasma        RCC     T-cell        pDC       NA's 
         0          0          0          0          0         56

In [78]:
dim(mica_input)

[1] 10680 15278

In [33]:
mica_output_file <- file.path(mica_out_dir, "clustering_UMAP_euclidean_20_2.15.txt")

clustered.eset <- addMICAoutput(
  input_eset       = log2cpm.eset,
  mica_output_file = mica_output_file,
  visual_method    = "umap"   # or "tsne"
)

saveRDS(clustered.eset, file.path(scminer_dir, "DATA/clustered.rds"))

ERROR: Error in addMICAoutput(input_eset = log2cpm.eset, mica_output_file = mica_output_file, : The eset and the mica_output_file do not match. Please check and re-try.


In [16]:
log2cpm.eset

SparseExpressionSet (storageMode: environment)
assayData: 10680 features, 15277 samples 
  element names: exprs 
protocolData: none
phenoData
  sampleNames: 0 1 ... 22429 (15277 total)
  varLabels: cell_type spot_name ... CellID (11 total)
  varMetadata: labelDescription
featureData
  featureNames: ENSG00000018869 ENSG00000130775 ... ENSG00000211445
    (10680 total)
  fvarLabels: SYMBOL nCell
  fvarMetadata: labelDescription
experimentData: use 'experimentData(object)'
Annotation:  

In [17]:
n_cells

Samples 
  15277

In [19]:
nrow(log2cpm.eset)

Features 
   10680